In [2]:
import duckdb

result = duckdb.sql("""
    SELECT COUNT(*) AS total_rows
    FROM '../data/events_silver_dedup.parquet'
""")

result.show()

┌────────────┐
│ total_rows │
│   int64    │
├────────────┤
│  109820004 │
└────────────┘



In [3]:
result = duckdb.sql("""
    SELECT
        user_id,
        event_time,
        event_type,
        LAG(event_time) OVER (PARTITION BY user_id ORDER BY event_time) AS previous_event_time
    FROM '../data/events_silver_dedup.parquet'
    WHERE user_id = 576718698
    ORDER BY event_time
""")

result.show()

┌───────────┬─────────────────────┬────────────┬─────────────────────┐
│  user_id  │     event_time      │ event_type │ previous_event_time │
│   int32   │      timestamp      │  varchar   │      timestamp      │
├───────────┼─────────────────────┼────────────┼─────────────────────┤
│ 576718698 │ 2019-11-25 19:44:30 │ view       │ NULL                │
│ 576718698 │ 2019-11-25 19:44:58 │ cart       │ 2019-11-25 19:44:30 │
│ 576718698 │ 2019-11-25 19:45:01 │ cart       │ 2019-11-25 19:44:58 │
└───────────┴─────────────────────┴────────────┴─────────────────────┘



In [4]:
# Calculate the actual time gap between each event and the previous one
# (in minutes), then flag whether that gap is 30+ minutes - meaning
# a new session should start at this row.
# DATE_DIFF('minute', a, b) gives the difference between two timestamps,
# in minutes.

result = duckdb.sql("""
    SELECT
        user_id,
        event_time,
        event_type,
        LAG(event_time) OVER (PARTITION BY user_id ORDER BY event_time) AS previous_event_time,
        DATE_DIFF(
            'minute',
            LAG(event_time) OVER (PARTITION BY user_id ORDER BY event_time),
            event_time
        ) AS minutes_since_previous,
        CASE
            WHEN DATE_DIFF(
                'minute',
                LAG(event_time) OVER (PARTITION BY user_id ORDER BY event_time),
                event_time
            ) >= 30 THEN 1
            WHEN LAG(event_time) OVER (PARTITION BY user_id ORDER BY event_time) IS NULL THEN 1
            ELSE 0
        END AS is_new_session
    FROM '../data/events_silver_dedup.parquet'
    WHERE user_id = 576718698
    ORDER BY event_time
""")

result.show()

┌───────────┬─────────────────────┬────────────┬─────────────────────┬────────────────────────┬────────────────┐
│  user_id  │     event_time      │ event_type │ previous_event_time │ minutes_since_previous │ is_new_session │
│   int32   │      timestamp      │  varchar   │      timestamp      │         int64          │     int32      │
├───────────┼─────────────────────┼────────────┼─────────────────────┼────────────────────────┼────────────────┤
│ 576718698 │ 2019-11-25 19:44:30 │ view       │ NULL                │                   NULL │              1 │
│ 576718698 │ 2019-11-25 19:44:58 │ cart       │ 2019-11-25 19:44:30 │                      0 │              0 │
│ 576718698 │ 2019-11-25 19:45:01 │ cart       │ 2019-11-25 19:44:58 │                      1 │              0 │
└───────────┴─────────────────────┴────────────┴─────────────────────┴────────────────────────┴────────────────┘



In [5]:
# Turn the is_new_session flags into an actual session number, by taking
# a RUNNING SUM of the flag, per user. Every time is_new_session = 1,
# the running total increases by 1 - effectively counting "how many
# sessions has this user started so far, up to and including this row."

result = duckdb.sql("""
    WITH gaps AS (
        SELECT
            user_id,
            event_time,
            event_type,
            CASE
                WHEN DATE_DIFF(
                    'minute',
                    LAG(event_time) OVER (PARTITION BY user_id ORDER BY event_time),
                    event_time
                ) >= 30 THEN 1
                WHEN LAG(event_time) OVER (PARTITION BY user_id ORDER BY event_time) IS NULL THEN 1
                ELSE 0
            END AS is_new_session
        FROM '../data/events_silver_dedup.parquet'
        WHERE user_id = 576718698
    )
    SELECT
        user_id,
        event_time,
        event_type,
        SUM(is_new_session) OVER (PARTITION BY user_id ORDER BY event_time) AS session_number
    FROM gaps
    ORDER BY event_time
""")

result.show()

┌───────────┬─────────────────────┬────────────┬────────────────┐
│  user_id  │     event_time      │ event_type │ session_number │
│   int32   │      timestamp      │  varchar   │     int128     │
├───────────┼─────────────────────┼────────────┼────────────────┤
│ 576718698 │ 2019-11-25 19:44:30 │ view       │              1 │
│ 576718698 │ 2019-11-25 19:44:58 │ cart       │              1 │
│ 576718698 │ 2019-11-25 19:45:01 │ cart       │              1 │
└───────────┴─────────────────────┴────────────┴────────────────┘



In [6]:
# Find a user with a decent number of events spread over multiple days,
# so we have a good test case to verify session_number actually increments.

result = duckdb.sql("""
    SELECT
        user_id,
        COUNT(*) AS event_count,
        COUNT(DISTINCT CAST(event_time AS DATE)) AS distinct_days
    FROM '../data/events_silver_dedup.parquet'
    GROUP BY user_id
    HAVING COUNT(DISTINCT CAST(event_time AS DATE)) >= 5
    ORDER BY event_count DESC
    LIMIT 5
""")

result.show()

┌───────────┬─────────────┬───────────────┐
│  user_id  │ event_count │ distinct_days │
│   int32   │    int64    │     int64     │
├───────────┼─────────────┼───────────────┤
│ 568778435 │       22925 │            19 │
│ 569335945 │       14810 │            20 │
│ 512475445 │       13547 │            59 │
│ 512365995 │       10055 │            61 │
│ 568818636 │        6171 │            15 │
└───────────┴─────────────┴───────────────┘



In [7]:
result = duckdb.sql("""
    WITH gaps AS (
        SELECT
            user_id,
            event_time,
            event_type,
            CASE
                WHEN DATE_DIFF(
                    'minute',
                    LAG(event_time) OVER (PARTITION BY user_id ORDER BY event_time),
                    event_time
                ) >= 30 THEN 1
                WHEN LAG(event_time) OVER (PARTITION BY user_id ORDER BY event_time) IS NULL THEN 1
                ELSE 0
            END AS is_new_session
        FROM '../data/events_silver_dedup.parquet'
        WHERE user_id = 512365995
    )
    SELECT
        user_id,
        event_time,
        event_type,
        SUM(is_new_session) OVER (PARTITION BY user_id ORDER BY event_time) AS session_number
    FROM gaps
    ORDER BY event_time
    LIMIT 30
""")

result.show()

┌───────────┬─────────────────────┬────────────┬────────────────┐
│  user_id  │     event_time      │ event_type │ session_number │
│   int32   │      timestamp      │  varchar   │     int128     │
├───────────┼─────────────────────┼────────────┼────────────────┤
│ 512365995 │ 2019-10-01 07:21:29 │ view       │              1 │
│ 512365995 │ 2019-10-01 07:22:48 │ view       │              1 │
│ 512365995 │ 2019-10-01 07:23:38 │ view       │              1 │
│ 512365995 │ 2019-10-01 07:24:06 │ view       │              1 │
│ 512365995 │ 2019-10-01 07:24:41 │ view       │              1 │
│ 512365995 │ 2019-10-01 07:24:50 │ view       │              1 │
│ 512365995 │ 2019-10-01 07:26:58 │ view       │              1 │
│ 512365995 │ 2019-10-01 07:28:28 │ view       │              1 │
│ 512365995 │ 2019-10-01 08:28:20 │ view       │              2 │
│ 512365995 │ 2019-10-01 08:32:46 │ view       │              2 │
│     ·     │          ·          │  ·         │              · │
│     ·   

In [8]:
# Run our session-building logic across the FULL dataset (all users, all rows).
# We also create a single globally-unique session identifier by combining
# user_id and session_number - e.g. "512365995_1" - since session_number alone
# repeats across different users (everyone's first session is "1").

duckdb.sql("""
    COPY (
        WITH gaps AS (
            SELECT
                *,
                CASE
                    WHEN DATE_DIFF(
                        'minute',
                        LAG(event_time) OVER (PARTITION BY user_id ORDER BY event_time),
                        event_time
                    ) >= 30 THEN 1
                    WHEN LAG(event_time) OVER (PARTITION BY user_id ORDER BY event_time) IS NULL THEN 1
                    ELSE 0
                END AS is_new_session
            FROM '../data/events_silver_dedup.parquet'
        ),
        numbered AS (
            SELECT
                *,
                SUM(is_new_session) OVER (PARTITION BY user_id ORDER BY event_time) AS session_number
            FROM gaps
        )
        SELECT
            * EXCLUDE (is_new_session, session_number),
            CAST(user_id AS VARCHAR) || '_' || CAST(session_number AS VARCHAR) AS rebuilt_session_id
        FROM numbered
    )
    TO '../data/events_with_sessions.parquet'
    (FORMAT PARQUET)
""")

print("Done building sessions across full dataset")

Done building sessions across full dataset


In [9]:
# Verify: row count should be unchanged from events_silver_dedup.parquet
# since we only ADDED a column, never filtered or dropped rows.

result = duckdb.sql("""
    SELECT
        (SELECT COUNT(*) FROM '../data/events_silver_dedup.parquet') AS original_rows,
        (SELECT COUNT(*) FROM '../data/events_with_sessions.parquet') AS with_sessions_rows,
        (SELECT COUNT(DISTINCT rebuilt_session_id) FROM '../data/events_with_sessions.parquet') AS distinct_rebuilt_sessions,
        (SELECT COUNT(DISTINCT user_session) FROM '../data/events_with_sessions.parquet') AS distinct_original_sessions
""")

result.show()

┌───────────────┬────────────────────┬───────────────────────────┬────────────────────────────┐
│ original_rows │ with_sessions_rows │ distinct_rebuilt_sessions │ distinct_original_sessions │
│     int64     │       int64        │           int64           │           int64            │
├───────────────┼────────────────────┼───────────────────────────┼────────────────────────────┤
│     109820004 │          109820004 │                  18776366 │                   23016650 │
└───────────────┴────────────────────┴───────────────────────────┴────────────────────────────┘



In [10]:
# Find cases where the ORIGINAL user_session changes, but very little time
# has passed since the previous event - i.e., cases where the original
# system started a "new" session even though a human would still call it
# the same visit.

result = duckdb.sql("""
    SELECT
        user_id,
        event_time,
        user_session,
        LAG(user_session) OVER (PARTITION BY user_id ORDER BY event_time) AS previous_session,
        DATE_DIFF(
            'minute',
            LAG(event_time) OVER (PARTITION BY user_id ORDER BY event_time),
            event_time
        ) AS minutes_since_previous
    FROM '../data/events_with_sessions.parquet'
    WHERE user_id = 512365995
    ORDER BY event_time
    LIMIT 40
""")

result.show()

┌───────────┬─────────────────────┬──────────────────────────────────────┬──────────────────────────────────────┬────────────────────────┐
│  user_id  │     event_time      │             user_session             │           previous_session           │ minutes_since_previous │
│   int32   │      timestamp      │               varchar                │               varchar                │         int64          │
├───────────┼─────────────────────┼──────────────────────────────────────┼──────────────────────────────────────┼────────────────────────┤
│ 512365995 │ 2019-10-01 07:21:29 │ be888c14-9731-4de2-9397-5e892475fba3 │ NULL                                 │                   NULL │
│ 512365995 │ 2019-10-01 07:22:48 │ be888c14-9731-4de2-9397-5e892475fba3 │ be888c14-9731-4de2-9397-5e892475fba3 │                      1 │
│ 512365995 │ 2019-10-01 07:23:38 │ be888c14-9731-4de2-9397-5e892475fba3 │ be888c14-9731-4de2-9397-5e892475fba3 │                      1 │
│ 512365995 │ 2019-10-01 07

In [11]:
# Find sessions that contain more than one purchase event.
# This is our test case: does grouping purchases by session actually
# make sense as an "order" definition, or does it look wrong up close?

result = duckdb.sql("""
    SELECT
        rebuilt_session_id,
        COUNT(*) AS purchase_count
    FROM '../data/events_with_sessions.parquet'
    WHERE event_type = 'purchase'
    GROUP BY rebuilt_session_id
    HAVING COUNT(*) > 1
    ORDER BY purchase_count DESC
    LIMIT 5
""")

result.show()

┌────────────────────┬────────────────┐
│ rebuilt_session_id │ purchase_count │
│      varchar       │     int64      │
├────────────────────┼────────────────┤
│ 560859100_25       │             89 │
│ 564068124_28       │             84 │
│ 518514099_27       │             76 │
│ 519023956_14       │             64 │
│ 568105973_6        │             55 │
└────────────────────┴────────────────┘



In [12]:
# Zoom into the session with 89 purchase events and look at the actual rows.
# Are these genuinely 89 different products being bought, or does something
# look repetitive/suspicious - similar to what we found with cart duplicates?

result = duckdb.sql("""
    SELECT
        event_time,
        product_id,
        category_code,
        price,
        user_id
    FROM '../data/events_with_sessions.parquet'
    WHERE rebuilt_session_id = '560859100_25'
      AND event_type = 'purchase'
    ORDER BY event_time
    LIMIT 20
""")

result.show()

┌─────────────────────┬────────────┬───────────────────────────────┬────────┬───────────┐
│     event_time      │ product_id │         category_code         │ price  │  user_id  │
│      timestamp      │   int32    │            varchar            │ float  │   int32   │
├─────────────────────┼────────────┼───────────────────────────────┼────────┼───────────┤
│ 2019-11-17 13:48:00 │    1701452 │ computers.peripherals.monitor │  94.96 │ 560859100 │
│ 2019-11-17 13:50:12 │    1701554 │ computers.peripherals.monitor │ 130.25 │ 560859100 │
│ 2019-11-17 13:53:01 │    1701447 │ computers.peripherals.monitor │ 122.87 │ 560859100 │
│ 2019-11-17 13:54:10 │    1701519 │ computers.peripherals.monitor │  84.81 │ 560859100 │
│ 2019-11-17 13:55:02 │    1701519 │ computers.peripherals.monitor │  84.81 │ 560859100 │
│ 2019-11-17 13:57:43 │    1700652 │ computers.peripherals.monitor │ 102.45 │ 560859100 │
│ 2019-11-17 13:59:58 │    1701509 │ computers.peripherals.monitor │  93.44 │ 560859100 │
│ 2019-11-

In [13]:
# Look at the TIME GAPS between consecutive purchase events within this
# one session, to see if they cluster into smaller bursts rather than
# being one continuous purchasing spree.

result = duckdb.sql("""
    SELECT
        event_time,
        product_id,
        price,
        DATE_DIFF(
            'minute',
            LAG(event_time) OVER (ORDER BY event_time),
            event_time
        ) AS minutes_since_previous_purchase
    FROM '../data/events_with_sessions.parquet'
    WHERE rebuilt_session_id = '560859100_25'
      AND event_type = 'purchase'
    ORDER BY event_time
""")

result.show()

┌─────────────────────┬────────────┬────────┬─────────────────────────────────┐
│     event_time      │ product_id │ price  │ minutes_since_previous_purchase │
│      timestamp      │   int32    │ float  │              int64              │
├─────────────────────┼────────────┼────────┼─────────────────────────────────┤
│ 2019-11-17 13:48:00 │    1701452 │  94.96 │                            NULL │
│ 2019-11-17 13:50:12 │    1701554 │ 130.25 │                               2 │
│ 2019-11-17 13:53:01 │    1701447 │ 122.87 │                               3 │
│ 2019-11-17 13:54:10 │    1701519 │  84.81 │                               1 │
│ 2019-11-17 13:55:02 │    1701519 │  84.81 │                               1 │
│ 2019-11-17 13:57:43 │    1700652 │ 102.45 │                               2 │
│ 2019-11-17 13:59:58 │    1701509 │  93.44 │                               2 │
│ 2019-11-17 14:01:11 │    1701369 │  92.02 │                               2 │
│ 2019-11-17 14:02:10 │    1701326 │  90

In [14]:
# Check the distribution of purchase-events-per-session across the WHOLE
# dataset, not just our one example - to see how common very-high-purchase
# sessions actually are.

result = duckdb.sql("""
    SELECT
        purchase_count,
        COUNT(*) AS num_sessions
    FROM (
        SELECT
            rebuilt_session_id,
            COUNT(*) AS purchase_count
        FROM '../data/events_with_sessions.parquet'
        WHERE event_type = 'purchase'
        GROUP BY rebuilt_session_id
    )
    GROUP BY purchase_count
    ORDER BY purchase_count DESC
    LIMIT 20
""")

result.show()

┌────────────────┬──────────────┐
│ purchase_count │ num_sessions │
│     int64      │    int64     │
├────────────────┼──────────────┤
│             89 │            1 │
│             84 │            1 │
│             76 │            1 │
│             64 │            1 │
│             55 │            1 │
│             54 │            1 │
│             52 │            2 │
│             51 │            1 │
│             49 │            1 │
│             47 │            1 │
│             46 │            1 │
│             45 │            1 │
│             44 │            2 │
│             42 │            3 │
│             41 │            1 │
│             40 │            1 │
│             39 │            1 │
│             37 │            1 │
│             36 │            1 │
│             35 │            1 │
└────────────────┴──────────────┘
  20 rows             2 columns



In [15]:
# Get the FULL distribution shape - specifically, how many sessions have
# just 1, 2, or 3 purchases (the expected normal range), compared to
# the total number of purchase-containing sessions overall.

result = duckdb.sql("""
    SELECT
        CASE
            WHEN purchase_count = 1 THEN '1'
            WHEN purchase_count = 2 THEN '2'
            WHEN purchase_count = 3 THEN '3'
            WHEN purchase_count BETWEEN 4 AND 10 THEN '4-10'
            WHEN purchase_count BETWEEN 11 AND 20 THEN '11-20'
            ELSE '21+'
        END AS purchase_count_bucket,
        COUNT(*) AS num_sessions,
        SUM(purchase_count) AS total_purchase_events
    FROM (
        SELECT
            rebuilt_session_id,
            COUNT(*) AS purchase_count
        FROM '../data/events_with_sessions.parquet'
        WHERE event_type = 'purchase'
        GROUP BY rebuilt_session_id
    )
    GROUP BY purchase_count_bucket
    ORDER BY MIN(purchase_count)
""")

result.show()

┌───────────────────────┬──────────────┬───────────────────────┐
│ purchase_count_bucket │ num_sessions │ total_purchase_events │
│        varchar        │    int64     │        int128         │
├───────────────────────┼──────────────┼───────────────────────┤
│ 1                     │      1061191 │               1061191 │
│ 2                     │       160334 │                320668 │
│ 3                     │        40709 │                122127 │
│ 4-10                  │        27767 │                138563 │
│ 11-20                 │          963 │                 12962 │
│ 21+                   │          144 │                  4192 │
└───────────────────────┴──────────────┴───────────────────────┘



In [16]:
# Build our order-level view: take all purchase events, tag each with:
# - order_id (= the session it happened in, since that's our order definition)
# - a flag for sessions with an unusually high purchase count (20+),
#   based on the natural drop-off point we found in the distribution

duckdb.sql("""
    COPY (
        WITH purchase_counts AS (
            SELECT
                rebuilt_session_id,
                COUNT(*) AS purchases_in_session
            FROM '../data/events_with_sessions.parquet'
            WHERE event_type = 'purchase'
            GROUP BY rebuilt_session_id
        )
        SELECT
            e.*,
            e.rebuilt_session_id AS order_id,
            pc.purchases_in_session,
            CASE WHEN pc.purchases_in_session >= 20 THEN 1 ELSE 0 END AS is_likely_non_standard_order
        FROM '../data/events_with_sessions.parquet' e
        INNER JOIN purchase_counts pc
            ON e.rebuilt_session_id = pc.rebuilt_session_id
        WHERE e.event_type = 'purchase'
    )
    TO '../data/purchases_with_orders.parquet'
    (FORMAT PARQUET)
""")

print("Done building purchases_with_orders.parquet")

Done building purchases_with_orders.parquet


In [17]:
result = duckdb.sql("""
    SELECT
        is_likely_non_standard_order,
        COUNT(*) AS num_rows,
        COUNT(DISTINCT order_id) AS num_orders
    FROM '../data/purchases_with_orders.parquet'
    GROUP BY is_likely_non_standard_order
""")

result.show()

┌──────────────────────────────┬──────────┬────────────┐
│ is_likely_non_standard_order │ num_rows │ num_orders │
│            int32             │  int64   │   int64    │
├──────────────────────────────┼──────────┼────────────┤
│                            1 │     4792 │        174 │
│                            0 │  1654911 │    1290934 │
└──────────────────────────────┴──────────┴────────────┘



In [18]:
# Double-check: does purchases_with_orders.parquet have the same total 
# purchase-event count as our original data? If the join went wrong, 
# we might have accidentally duplicated some rows.

result = duckdb.sql("""
    SELECT
        (SELECT COUNT(*) FROM '../data/events_with_sessions.parquet' WHERE event_type = 'purchase') AS original_purchase_events,
        (SELECT COUNT(*) FROM '../data/purchases_with_orders.parquet') AS new_purchase_events
""")

result.show()

┌──────────────────────────┬─────────────────────┐
│ original_purchase_events │ new_purchase_events │
│          int64           │        int64        │
├──────────────────────────┼─────────────────────┤
│                  1659703 │             1659703 │
└──────────────────────────┴─────────────────────┘



In [19]:
# Check specifically: how many sessions have EXACTLY 20 purchases?
# If this bridges the gap between 144 and 174, that confirms the 
# discrepancy is just a threshold boundary difference, not a real bug.

result = duckdb.sql("""
    SELECT COUNT(*) AS sessions_with_exactly_20
    FROM (
        SELECT rebuilt_session_id, COUNT(*) AS purchase_count
        FROM '../data/events_with_sessions.parquet'
        WHERE event_type = 'purchase'
        GROUP BY rebuilt_session_id
    )
    WHERE purchase_count = 20
""")

result.show()

┌──────────────────────────┐
│ sessions_with_exactly_20 │
│          int64           │
├──────────────────────────┤
│                       30 │
└──────────────────────────┘



In [20]:
# Find orders (sessions) where the SAME product_id appears more than once
# among purchase events. We'll look at the actual rows to judge whether
# this looks like genuine repeat-buying or duplicate tracking.

result = duckdb.sql("""
    SELECT
        order_id,
        product_id,
        COUNT(*) AS times_purchased,
        MIN(event_time) AS first_purchase,
        MAX(event_time) AS last_purchase
    FROM '../data/purchases_with_orders.parquet'
    WHERE is_likely_non_standard_order = 0
    GROUP BY order_id, product_id
    HAVING COUNT(*) > 1
    ORDER BY times_purchased DESC
    LIMIT 10
""")

result.show()

┌──────────────┬────────────┬─────────────────┬─────────────────────┬─────────────────────┐
│   order_id   │ product_id │ times_purchased │   first_purchase    │    last_purchase    │
│   varchar    │   int32    │      int64      │      timestamp      │      timestamp      │
├──────────────┼────────────┼─────────────────┼─────────────────────┼─────────────────────┤
│ 519311739_6  │    3701432 │              18 │ 2019-10-18 12:34:48 │ 2019-10-18 12:55:44 │
│ 548318522_4  │    1005115 │              18 │ 2019-10-01 18:27:14 │ 2019-10-01 19:17:14 │
│ 564749032_1  │    1004767 │              18 │ 2019-10-27 15:37:40 │ 2019-10-27 15:57:32 │
│ 517815158_48 │   12300181 │              18 │ 2019-10-13 07:36:35 │ 2019-10-13 08:01:15 │
│ 518266288_56 │   11400317 │              18 │ 2019-11-27 10:28:11 │ 2019-11-27 10:39:47 │
│ 517815158_92 │   15800131 │              17 │ 2019-10-28 05:29:59 │ 2019-10-28 05:53:03 │
│ 513219116_6  │    1004249 │              17 │ 2019-10-14 15:10:37 │ 2019-10-14

In [21]:
# For every order, check: how many rows are "extra" repeats of a product
# already purchased earlier in that same order? 
# (i.e., 2nd, 3rd, 4th... occurrence of the same product in the same order)

result = duckdb.sql("""
    WITH ranked AS (
        SELECT
            order_id,
            product_id,
            ROW_NUMBER() OVER (PARTITION BY order_id, product_id ORDER BY event_time) AS occurrence_number
        FROM '../data/purchases_with_orders.parquet'
        WHERE is_likely_non_standard_order = 0
    )
    SELECT
        COUNT(*) AS total_purchase_rows,
        SUM(CASE WHEN occurrence_number = 1 THEN 1 ELSE 0 END) AS first_occurrences,
        SUM(CASE WHEN occurrence_number > 1 THEN 1 ELSE 0 END) AS repeat_occurrences
    FROM ranked
""")

result.show()

┌─────────────────────┬───────────────────┬────────────────────┐
│ total_purchase_rows │ first_occurrences │ repeat_occurrences │
│        int64        │      int128       │       int128       │
├─────────────────────┼───────────────────┼────────────────────┤
│             1654911 │           1502219 │             152692 │
└─────────────────────┴───────────────────┴────────────────────┘



In [22]:
# Compare total revenue under two interpretations:
# 1. "Quantity" interpretation: every purchase row counts, so repeats = buying more units
# 2. "Duplicate" interpretation: only the FIRST occurrence of each product per order counts,
#    repeats are treated as tracking artifacts and excluded

result = duckdb.sql("""
    WITH ranked AS (
        SELECT
            order_id,
            product_id,
            price,
            ROW_NUMBER() OVER (PARTITION BY order_id, product_id ORDER BY event_time) AS occurrence_number
        FROM '../data/purchases_with_orders.parquet'
        WHERE is_likely_non_standard_order = 0
    )
    SELECT
        SUM(price) AS revenue_if_all_counted_as_quantity,
        SUM(CASE WHEN occurrence_number = 1 THEN price ELSE 0 END) AS revenue_if_only_first_occurrence_counted
    FROM ranked
""")

result.show()

┌────────────────────────────────────┬──────────────────────────────────────────┐
│ revenue_if_all_counted_as_quantity │ revenue_if_only_first_occurrence_counted │
│               double               │                  double                  │
├────────────────────────────────────┼──────────────────────────────────────────┤
│                  503079668.2588908 │                        451263778.5306107 │
└────────────────────────────────────┴──────────────────────────────────────────┘



In [23]:
# Apply the "first occurrence only" rule: for each (order_id, product_id) pair,
# keep only the first purchase event chronologically, drop any repeats.
# This treats repeated purchases of the same product within one order as
# duplicate tracking events, not genuine additional quantity.

duckdb.sql("""
    COPY (
        WITH ranked AS (
            SELECT
                *,
                ROW_NUMBER() OVER (PARTITION BY order_id, product_id ORDER BY event_time) AS occurrence_number
            FROM '../data/purchases_with_orders.parquet'
        )
        SELECT * EXCLUDE (occurrence_number)
        FROM ranked
        WHERE occurrence_number = 1
    )
    TO '../data/purchases_final.parquet'
    (FORMAT PARQUET)
""")

print("Done applying first-occurrence rule")

Done applying first-occurrence rule


In [24]:
# Verify: row count should now equal our earlier "first_occurrences" count (1,502,219)
# plus whatever first-occurrences existed among the non-standard orders we excluded before

result = duckdb.sql("""
    SELECT
        COUNT(*) AS final_row_count,
        SUM(price) AS final_total_revenue
    FROM '../data/purchases_final.parquet'
""")

result.show()

┌─────────────────┬─────────────────────┐
│ final_row_count │ final_total_revenue │
│      int64      │       double        │
├─────────────────┼─────────────────────┤
│         1503806 │   451807232.5601831 │
└─────────────────┴─────────────────────┘



In [25]:
result = duckdb.sql("""
    WITH ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (PARTITION BY order_id, product_id ORDER BY event_time) AS occurrence_number
        FROM '../data/purchases_with_orders.parquet'
        WHERE is_likely_non_standard_order = 1
    )
    SELECT COUNT(*) AS first_occurrences_in_nonstandard_orders
    FROM ranked
    WHERE occurrence_number = 1
""")

result.show()

┌─────────────────────────────────────────┐
│ first_occurrences_in_nonstandard_orders │
│                  int64                  │
├─────────────────────────────────────────┤
│                                    1587 │
└─────────────────────────────────────────┘



In [2]:
# Build dim_date: one row per calendar date in our data's range,
# with useful descriptive attributes pre-computed.
# generate_series() creates a sequence of dates between two points.
# DAYNAME/MONTHNAME give text labels; DAYOFWEEK gives a number (0=Sunday typically in DuckDB).
import duckdb
duckdb.sql("""
    COPY (
        SELECT
            CAST(d AS DATE) AS date_key,
            EXTRACT(YEAR FROM d) AS year,
            EXTRACT(MONTH FROM d) AS month,
            MONTHNAME(d) AS month_name,
            EXTRACT(DAY FROM d) AS day_of_month,
            DAYNAME(d) AS day_name,
            EXTRACT(DOW FROM d) AS day_of_week_number,
            CASE WHEN EXTRACT(DOW FROM d) IN (0, 6) THEN TRUE ELSE FALSE END AS is_weekend,
            CASE WHEN CAST(d AS DATE) BETWEEN '2019-11-15' AND '2019-11-19' THEN TRUE ELSE FALSE END AS is_promo_period
        FROM generate_series(DATE '2019-10-01', DATE '2019-11-30', INTERVAL 1 DAY) AS t(d)
    )
    TO '../data/dim_date.parquet'
    (FORMAT PARQUET)
""")

print("Done building dim_date")

Done building dim_date


In [3]:
# Verify dim_date: check row count, and spot-check a few known dates
# to confirm day-of-week and weekend logic is correct.
# 2019-10-05 was a Saturday, 2019-10-06 was a Sunday - real-world facts
# we can check our output against.

result = duckdb.sql("""
    SELECT
        date_key,
        day_name,
        day_of_week_number,
        is_weekend,
        is_promo_period
    FROM '../data/dim_date.parquet'
    WHERE date_key IN ('2019-10-04', '2019-10-05', '2019-10-06', '2019-10-07')
    ORDER BY date_key
""")

result.show()

count_result = duckdb.sql("SELECT COUNT(*) AS total_dates FROM '../data/dim_date.parquet'")
count_result.show()

┌────────────┬──────────┬────────────────────┬────────────┬─────────────────┐
│  date_key  │ day_name │ day_of_week_number │ is_weekend │ is_promo_period │
│    date    │ varchar  │       int64        │  boolean   │     boolean     │
├────────────┼──────────┼────────────────────┼────────────┼─────────────────┤
│ 2019-10-04 │ Friday   │                  5 │ false      │ false           │
│ 2019-10-05 │ Saturday │                  6 │ true       │ false           │
│ 2019-10-06 │ Sunday   │                  0 │ true       │ false           │
│ 2019-10-07 │ Monday   │                  1 │ false      │ false           │
└────────────┴──────────┴────────────────────┴────────────┴─────────────────┘

┌─────────────┐
│ total_dates │
│    int64    │
├─────────────┤
│          61 │
└─────────────┘



In [4]:
# Verify our hardcoded promo period actually matches where the real spike occurred.
# Pull the daily view counts around that window from our events table,
# so we can compare exact numbers against our assumed date range.

result = duckdb.sql("""
    SELECT
        CAST(event_time AS DATE) AS event_date,
        COUNT(*) AS view_count
    FROM '../data/events_with_sessions.parquet'
    WHERE event_type = 'view'
      AND CAST(event_time AS DATE) BETWEEN '2019-11-12' AND '2019-11-22'
    GROUP BY event_date
    ORDER BY event_date
""")

result.show()

┌────────────┬────────────┐
│ event_date │ view_count │
│    date    │   int64    │
├────────────┼────────────┤
│ 2019-11-12 │    1895060 │
│ 2019-11-13 │    1924916 │
│ 2019-11-14 │    2877071 │
│ 2019-11-15 │    5737078 │
│ 2019-11-16 │    6027799 │
│ 2019-11-17 │    5783122 │
│ 2019-11-18 │    1909738 │
│ 2019-11-19 │    1630977 │
│ 2019-11-20 │    1602680 │
│ 2019-11-21 │    1576425 │
│ 2019-11-22 │    1473785 │
└────────────┴────────────┘
  11 rows       2 columns



In [5]:
# Rebuild dim_date with the CORRECTED promo period (Nov 14-17), based on
# actual daily view counts rather than an eyeballed chart estimate.

duckdb.sql("""
    COPY (
        SELECT
            CAST(d AS DATE) AS date_key,
            EXTRACT(YEAR FROM d) AS year,
            EXTRACT(MONTH FROM d) AS month,
            MONTHNAME(d) AS month_name,
            EXTRACT(DAY FROM d) AS day_of_month,
            DAYNAME(d) AS day_name,
            EXTRACT(DOW FROM d) AS day_of_week_number,
            CASE WHEN EXTRACT(DOW FROM d) IN (0, 6) THEN TRUE ELSE FALSE END AS is_weekend,
            CASE WHEN CAST(d AS DATE) BETWEEN '2019-11-14' AND '2019-11-17' THEN TRUE ELSE FALSE END AS is_promo_period
        FROM generate_series(DATE '2019-10-01', DATE '2019-11-30', INTERVAL 1 DAY) AS t(d)
    )
    TO '../data/dim_date.parquet'
    (FORMAT PARQUET)
""")

print("Done rebuilding dim_date with corrected promo window")

Done rebuilding dim_date with corrected promo window


In [6]:
# Check how many dot-separated levels exist in category_code values.
# LENGTH(...) - LENGTH(REPLACE(..., '.', '')) counts how many dots are in
# the string; number of levels = number of dots + 1.

result = duckdb.sql("""
    SELECT
        LENGTH(category_code) - LENGTH(REPLACE(category_code, '.', '')) + 1 AS num_levels,
        COUNT(*) AS row_count,
        COUNT(DISTINCT category_code) AS distinct_categories
    FROM '../data/events_with_sessions.parquet'
    WHERE category_code NOT LIKE 'unknown_%'
    GROUP BY num_levels
    ORDER BY num_levels
""")

result.show()

┌────────────┬───────────┬─────────────────────┐
│ num_levels │ row_count │ distinct_categories │
│   int64    │   int64   │        int64        │
├────────────┼───────────┼─────────────────────┤
│          2 │  44074473 │                  42 │
│          3 │  30262364 │                  86 │
│          4 │     99482 │                   1 │
└────────────┴───────────┴─────────────────────┘



In [7]:
# Look at the one 4-level category to understand what it actually is
result = duckdb.sql("""
    SELECT DISTINCT category_code
    FROM '../data/events_with_sessions.parquet'
    WHERE LENGTH(category_code) - LENGTH(REPLACE(category_code, '.', '')) + 1 = 4
""")

result.show()

┌─────────────────────────────────────┐
│            category_code            │
│               varchar               │
├─────────────────────────────────────┤
│ electronics.audio.music_tools.piano │
└─────────────────────────────────────┘



In [8]:
# Build dim_category: one row per distinct category_code (including our
# "unknown_" ones), split into up to 4 hierarchy levels.
# SPLIT_PART(string, delimiter, position) extracts one piece of a delimited
# string - e.g. SPLIT_PART('a.b.c', '.', 2) returns 'b'.
# If a category doesn't have that many levels, SPLIT_PART returns an empty
# string, which we convert to NULL for cleanliness.

duckdb.sql("""
    COPY (
        SELECT DISTINCT
            category_code,
            category_id,
            NULLIF(SPLIT_PART(category_code, '.', 1), '') AS category_level_1,
            NULLIF(SPLIT_PART(category_code, '.', 2), '') AS category_level_2,
            NULLIF(SPLIT_PART(category_code, '.', 3), '') AS category_level_3,
            NULLIF(SPLIT_PART(category_code, '.', 4), '') AS category_level_4
        FROM '../data/events_with_sessions.parquet'
    )
    TO '../data/dim_category.parquet'
    (FORMAT PARQUET)
""")

print("Done building dim_category")

Done building dim_category


In [9]:
# Verify category_id is truly unique in dim_category - i.e., it's a valid
# primary key candidate, with no duplicate rows for the same category_id.

result = duckdb.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT category_id) AS distinct_category_ids
    FROM '../data/dim_category.parquet'
""")

result.show()

┌────────────┬───────────────────────┐
│ total_rows │ distinct_category_ids │
│   int64    │         int64         │
├────────────┼───────────────────────┤
│        691 │                   691 │
└────────────┴───────────────────────┘



In [10]:
# Break down the 691 total categories: how many are "real" coded categories
# vs. our own "unknown_" synthetic fill-ins?

result = duckdb.sql("""
    SELECT
        CASE WHEN category_code LIKE 'unknown_%' THEN 'unknown (synthetic)' ELSE 'real (coded)' END AS category_type,
        COUNT(*) AS num_categories
    FROM '../data/dim_category.parquet'
    GROUP BY category_type
""")

result.show()

┌─────────────────────┬────────────────┐
│    category_type    │ num_categories │
│       varchar       │     int64      │
├─────────────────────┼────────────────┤
│ unknown (synthetic) │            414 │
│ real (coded)        │            277 │
└─────────────────────┴────────────────┘



In [11]:
result = duckdb.sql("""
    SELECT COUNT(DISTINCT category_code) AS distinct_real_categories
    FROM '../data/events_with_sessions.parquet'
    WHERE category_code NOT LIKE 'unknown_%'
""")

result.show()


┌──────────────────────────┐
│ distinct_real_categories │
│          int64           │
├──────────────────────────┤
│                      129 │
└──────────────────────────┘



In [12]:
# Check the REVERSE relationship: does each category_code map to exactly
# ONE category_id, or can the same readable category_code be associated
# with multiple different category_ids?

result = duckdb.sql("""
    SELECT
        category_code,
        COUNT(DISTINCT category_id) AS distinct_ids
    FROM '../data/events_with_sessions.parquet'
    WHERE category_code NOT LIKE 'unknown_%'
    GROUP BY category_code
    HAVING COUNT(DISTINCT category_id) > 1
    ORDER BY distinct_ids DESC
""")

result.show()

┌───────────────────────────────┬──────────────┐
│         category_code         │ distinct_ids │
│            varchar            │    int64     │
├───────────────────────────────┼──────────────┤
│ apparel.shoes                 │           27 │
│ apparel.costume               │            8 │
│ apparel.shoes.keds            │            8 │
│ accessories.bag               │            7 │
│ apparel.shirt                 │            7 │
│ sport.bicycle                 │            6 │
│ apparel.shoes.sandals         │            5 │
│ apparel.trousers              │            5 │
│ furniture.living_room.cabinet │            4 │
│ apparel.tshirt                │            4 │
│       ·                       │            · │
│       ·                       │            · │
│       ·                       │            · │
│ computers.notebook            │            2 │
│ electronics.video.tv          │            2 │
│ construction.tools.pump       │            2 │
│ construction.tools

In [13]:
# Spot-check: verify the 4-level category split correctly into all 4 columns
result = duckdb.sql("""
    SELECT
        category_id,
        category_code,
        category_level_1,
        category_level_2,
        category_level_3,
        category_level_4
    FROM '../data/dim_category.parquet'
    WHERE category_code = 'electronics.audio.music_tools.piano'
""")

result.show()

┌─────────────────────┬─────────────────────────────────────┬──────────────────┬──────────────────┬──────────────────┬──────────────────┐
│     category_id     │            category_code            │ category_level_1 │ category_level_2 │ category_level_3 │ category_level_4 │
│        int64        │               varchar               │     varchar      │     varchar      │     varchar      │     varchar      │
├─────────────────────┼─────────────────────────────────────┼──────────────────┼──────────────────┼──────────────────┼──────────────────┤
│ 2053013557603205653 │ electronics.audio.music_tools.piano │ electronics      │ audio            │ music_tools      │ piano            │
└─────────────────────┴─────────────────────────────────────┴──────────────────┴──────────────────┴──────────────────┴──────────────────┘



In [14]:
result = duckdb.sql("""
    SELECT
        category_id,
        category_code,
        category_level_1,
        category_level_2,
        category_level_3,
        category_level_4
    FROM '../data/dim_category.parquet'
    WHERE category_level_1 = 'electronics' AND category_level_3 IS NULL
    LIMIT 3
""")

result.show()

┌─────────────────────┬────────────────────┬──────────────────┬──────────────────┬──────────────────┬──────────────────┐
│     category_id     │   category_code    │ category_level_1 │ category_level_2 │ category_level_3 │ category_level_4 │
│        int64        │      varchar       │     varchar      │     varchar      │     varchar      │     varchar      │
├─────────────────────┼────────────────────┼──────────────────┼──────────────────┼──────────────────┼──────────────────┤
│ 2053013561579406073 │ electronics.clocks │ electronics      │ clocks           │ NULL             │ NULL             │
│ 2172371436436455782 │ electronics.tablet │ electronics      │ tablet           │ NULL             │ NULL             │
│ 2053013561059312345 │ electronics.tablet │ electronics      │ tablet           │ NULL             │ NULL             │
└─────────────────────┴────────────────────┴──────────────────┴──────────────────┴──────────────────┴──────────────────┘



In [15]:
# Find a product with several distinct prices over time - a good test case
# for building SCD Type 2 logic by hand first, before scaling to all products.

result = duckdb.sql("""
    SELECT
        product_id,
        COUNT(DISTINCT price) AS distinct_prices,
        COUNT(*) AS total_events
    FROM '../data/events_with_sessions.parquet'
    GROUP BY product_id
    HAVING COUNT(DISTINCT price) >= 3
    ORDER BY total_events DESC
    LIMIT 5
""")

result.show()

┌────────────┬─────────────────┬──────────────┐
│ product_id │ distinct_prices │ total_events │
│   int32    │      int64      │    int64     │
├────────────┼─────────────────┼──────────────┤
│    1004856 │              81 │      1130193 │
│    1005115 │             109 │      1024112 │
│    1004767 │             142 │      1004181 │
│    4804056 │              86 │       603233 │
│    1004870 │             161 │       535966 │
└────────────┴─────────────────┴──────────────┘



In [16]:
# Look at this product's actual price-over-time pattern.
# Group by date and price to see how often and how much the price moves.

result = duckdb.sql("""
    SELECT
        CAST(event_time AS DATE) AS event_date,
        price,
        COUNT(*) AS num_events
    FROM '../data/events_with_sessions.parquet'
    WHERE product_id = 1004856
    GROUP BY event_date, price
    ORDER BY event_date
    LIMIT 30
""")

result.show()

┌────────────┬────────┬────────────┐
│ event_date │ price  │ num_events │
│    date    │ float  │   int64    │
├────────────┼────────┼────────────┤
│ 2019-10-01 │ 132.87 │       3057 │
│ 2019-10-01 │ 130.76 │       3424 │
│ 2019-10-01 │ 130.25 │        585 │
│ 2019-10-01 │  130.7 │       6822 │
│ 2019-10-02 │ 132.87 │       3608 │
│ 2019-10-02 │ 130.25 │      10307 │
│ 2019-10-03 │ 130.76 │       1839 │
│ 2019-10-03 │ 132.31 │       3308 │
│ 2019-10-03 │ 130.25 │       2064 │
│ 2019-10-03 │ 132.15 │       3345 │
│     ·      │    ·   │         ·  │
│     ·      │    ·   │         ·  │
│     ·      │    ·   │         ·  │
│ 2019-10-06 │ 131.77 │       3378 │
│ 2019-10-07 │ 131.02 │       3497 │
│ 2019-10-07 │ 131.76 │       9763 │
│ 2019-10-07 │ 130.76 │        754 │
│ 2019-10-08 │ 130.76 │       2303 │
│ 2019-10-08 │ 131.02 │      12270 │
│ 2019-10-09 │ 131.02 │       7074 │
│ 2019-10-09 │ 130.49 │       7311 │
│ 2019-10-10 │ 130.47 │       9979 │
│ 2019-10-10 │ 130.49 │       6718 │
└

In [17]:
# For every product, count how many distinct prices it has ever had.
# This tells us the true scale of the problem before we decide on a
# simplification strategy.

result = duckdb.sql("""
    SELECT
        COUNT(DISTINCT product_id) AS total_products,
        SUM(distinct_prices) AS total_price_versions_if_tracked_literally,
        AVG(distinct_prices) AS avg_distinct_prices_per_product,
        MAX(distinct_prices) AS max_distinct_prices_for_one_product
    FROM (
        SELECT product_id, COUNT(DISTINCT price) AS distinct_prices
        FROM '../data/events_with_sessions.parquet'
        GROUP BY product_id
    )
""")

result.show()

┌────────────────┬───────────────────────────────────────────┬─────────────────────────────────┬─────────────────────────────────────┐
│ total_products │ total_price_versions_if_tracked_literally │ avg_distinct_prices_per_product │ max_distinct_prices_for_one_product │
│     int64      │                  int128                   │             double              │                int64                │
├────────────────┼───────────────────────────────────────────┼─────────────────────────────────┼─────────────────────────────────────┤
│         206876 │                                    595151 │              2.8768489336607437 │                                 183 │
└────────────────┴───────────────────────────────────────────┴─────────────────────────────────┴─────────────────────────────────────┘



In [18]:
# Test our proposed approach: what if we only track ONE price per product
# per day (e.g. the first price seen that day)? How much does this reduce
# the total number of price-version rows compared to the literal approach?

result = duckdb.sql("""
    WITH daily_price AS (
        SELECT DISTINCT
            product_id,
            CAST(event_time AS DATE) AS price_date,
            FIRST(price ORDER BY event_time) AS price
        FROM '../data/events_with_sessions.parquet'
        GROUP BY product_id, CAST(event_time AS DATE)
    )
    SELECT
        COUNT(*) AS total_daily_price_rows,
        COUNT(DISTINCT product_id) AS total_products
    FROM daily_price
""")

result.show()

┌────────────────────────┬────────────────┐
│ total_daily_price_rows │ total_products │
│         int64          │     int64      │
├────────────────────────┼────────────────┤
│                4998112 │         206876 │
└────────────────────────┴────────────────┘



In [19]:
# For our test product, find the actual price CHANGE POINTS - moments where
# the price differs from the immediately preceding price (in event-time order),
# not just "how many distinct prices exist" or "one row per day."
# This correctly collapses repeated/reappearing prices and intra-day noise
# into proper contiguous periods.

result = duckdb.sql("""
    SELECT
        event_time,
        price,
        LAG(price) OVER (ORDER BY event_time) AS previous_price,
        CASE WHEN price != LAG(price) OVER (ORDER BY event_time) THEN 1 ELSE 0 END AS price_changed
    FROM '../data/events_with_sessions.parquet'
    WHERE product_id = 1004856
    ORDER BY event_time
    LIMIT 20
""")

result.show()

┌─────────────────────┬────────┬────────────────┬───────────────┐
│     event_time      │ price  │ previous_price │ price_changed │
│      timestamp      │ float  │     float      │     int32     │
├─────────────────────┼────────┼────────────────┼───────────────┤
│ 2019-10-01 00:01:06 │ 130.76 │           NULL │             0 │
│ 2019-10-01 00:01:17 │ 130.76 │         130.76 │             0 │
│ 2019-10-01 00:01:46 │ 130.76 │         130.76 │             0 │
│ 2019-10-01 00:01:59 │ 130.76 │         130.76 │             0 │
│ 2019-10-01 00:02:14 │ 130.76 │         130.76 │             0 │
│ 2019-10-01 00:02:48 │ 130.76 │         130.76 │             0 │
│ 2019-10-01 00:03:02 │ 130.76 │         130.76 │             0 │
│ 2019-10-01 00:03:51 │ 130.76 │         130.76 │             0 │
│ 2019-10-01 00:05:48 │ 130.76 │         130.76 │             0 │
│ 2019-10-01 00:09:15 │ 130.76 │         130.76 │             0 │
│ 2019-10-01 00:09:22 │ 130.76 │         130.76 │             0 │
│ 2019-10-

In [20]:
# Find the actual moments where price_changed = 1 for this product -
# i.e., the real chronological change points, not just "distinct prices
# seen on a given calendar date" (which can be misleading if events are
# processed slightly out of order, or if multiple session snapshots
# get grouped by date rather than by true sequence).

result = duckdb.sql("""
    SELECT
        event_time,
        price,
        LAG(price) OVER (ORDER BY event_time) AS previous_price
    FROM '../data/events_with_sessions.parquet'
    WHERE product_id = 1004856
    QUALIFY price != LAG(price) OVER (ORDER BY event_time)
    ORDER BY event_time
    LIMIT 20
""")

result.show()

┌─────────────────────┬────────┬────────────────┐
│     event_time      │ price  │ previous_price │
│      timestamp      │ float  │     float      │
├─────────────────────┼────────┼────────────────┤
│ 2019-10-01 07:13:17 │  130.7 │         130.76 │
│ 2019-10-01 15:04:38 │ 132.87 │          130.7 │
│ 2019-10-01 18:57:45 │ 130.25 │         132.87 │
│ 2019-10-02 05:58:12 │ 132.87 │         130.25 │
│ 2019-10-02 10:03:51 │ 130.25 │         132.87 │
│ 2019-10-03 05:14:22 │ 132.31 │         130.25 │
│ 2019-10-03 09:21:53 │ 132.17 │         132.31 │
│ 2019-10-03 13:22:23 │ 132.15 │         132.17 │
│ 2019-10-03 16:58:54 │ 130.76 │         132.15 │
│ 2019-10-04 04:20:58 │ 130.51 │         130.76 │
│ 2019-10-04 08:18:06 │ 132.05 │         130.51 │
│ 2019-10-05 03:15:09 │ 131.92 │         132.05 │
│ 2019-10-05 11:59:55 │ 131.79 │         131.92 │
│ 2019-10-06 03:03:06 │ 131.77 │         131.79 │
│ 2019-10-06 07:12:16 │ 131.76 │         131.77 │
│ 2019-10-06 15:00:51 │ 130.76 │         131.76 │


In [21]:
# Build dim_product using SCD Type 2: one row per (product, price period).
# Step 1: find every price CHANGE POINT per product, using the same
# LAG()-comparison technique we just validated.
# Step 2: each change point becomes the START of a new version;
# the END of each version is simply the start of the NEXT version for
# that same product (or NULL/open-ended if it's the most recent version).

duckdb.sql("""
    COPY (
        WITH price_events AS (
            SELECT
                product_id,
                category_id,
                brand,
                price,
                event_time,
                LAG(price) OVER (PARTITION BY product_id ORDER BY event_time) AS previous_price
            FROM '../data/events_with_sessions.parquet'
        ),
        change_points AS (
            SELECT
                product_id,
                category_id,
                brand,
                price,
                event_time AS valid_from
            FROM price_events
            WHERE previous_price IS NULL OR price != previous_price
        ),
        versioned AS (
            SELECT
                product_id,
                category_id,
                brand,
                price,
                valid_from,
                LEAD(valid_from) OVER (PARTITION BY product_id ORDER BY valid_from) AS valid_to,
                ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY valid_from DESC) AS recency_rank
            FROM change_points
        )
        SELECT
            product_id,
            category_id,
            brand,
            price,
            valid_from,
            valid_to,
            CASE WHEN recency_rank = 1 THEN TRUE ELSE FALSE END AS is_current
        FROM versioned
    )
    TO '../data/dim_product.parquet'
    (FORMAT PARQUET)
""")

print("Done building dim_product with SCD Type 2 price history")

Done building dim_product with SCD Type 2 price history


In [22]:
# Verify dim_product: total rows, distinct products, and confirm
# each product has exactly one "current" version (not zero, not multiple).

result = duckdb.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT product_id) AS distinct_products,
        SUM(CASE WHEN is_current THEN 1 ELSE 0 END) AS total_current_flags
    FROM '../data/dim_product.parquet'
""")

result.show()

┌────────────┬───────────────────┬─────────────────────┐
│ total_rows │ distinct_products │ total_current_flags │
│   int64    │       int64       │       int128        │
├────────────┼───────────────────┼─────────────────────┤
│     777246 │            206876 │              206876 │
└────────────┴───────────────────┴─────────────────────┘



In [23]:
# Spot-check our familiar volatile-pricing product: does it show
# multiple price versions, correctly ordered, with proper valid_from/valid_to?

result = duckdb.sql("""
    SELECT
        product_id,
        price,
        valid_from,
        valid_to,
        is_current
    FROM '../data/dim_product.parquet'
    WHERE product_id = 1004856
    ORDER BY valid_from
    LIMIT 10
""")

result.show()

┌────────────┬────────┬─────────────────────┬─────────────────────┬────────────┐
│ product_id │ price  │     valid_from      │      valid_to       │ is_current │
│   int32    │ float  │      timestamp      │      timestamp      │  boolean   │
├────────────┼────────┼─────────────────────┼─────────────────────┼────────────┤
│    1004856 │ 130.76 │ 2019-10-01 00:01:06 │ 2019-10-01 07:13:17 │ false      │
│    1004856 │  130.7 │ 2019-10-01 07:13:17 │ 2019-10-01 15:04:38 │ false      │
│    1004856 │ 132.87 │ 2019-10-01 15:04:38 │ 2019-10-01 18:57:45 │ false      │
│    1004856 │ 130.25 │ 2019-10-01 18:57:45 │ 2019-10-02 05:58:12 │ false      │
│    1004856 │ 132.87 │ 2019-10-02 05:58:12 │ 2019-10-02 10:03:51 │ false      │
│    1004856 │ 130.25 │ 2019-10-02 10:03:51 │ 2019-10-03 05:14:22 │ false      │
│    1004856 │ 132.31 │ 2019-10-03 05:14:22 │ 2019-10-03 09:21:53 │ false      │
│    1004856 │ 132.17 │ 2019-10-03 09:21:53 │ 2019-10-03 13:22:23 │ false      │
│    1004856 │ 132.15 │ 2019

In [24]:
# Check 1: do users actually repeat-purchase? This determines whether
# cohort retention (Q4/Q8) is viable at all.
result = duckdb.sql("""
    SELECT
        orders_per_user,
        COUNT(*) AS num_users
    FROM (
        SELECT user_id, COUNT(DISTINCT order_id) AS orders_per_user
        FROM '../data/purchases_final.parquet'
        GROUP BY user_id
    )
    GROUP BY orders_per_user
    ORDER BY orders_per_user
    LIMIT 10
""")
result.show()

┌─────────────────┬───────────┐
│ orders_per_user │ num_users │
│      int64      │   int64   │
├─────────────────┼───────────┤
│               1 │    464424 │
│               2 │    125652 │
│               3 │     46840 │
│               4 │     21638 │
│               5 │     11628 │
│               6 │      7247 │
│               7 │      4684 │
│               8 │      3194 │
│               9 │      2328 │
│              10 │      1775 │
└─────────────────┴───────────┘
  10 rows           2 columns



In [25]:
# Check 2: how often is brand missing? Determines whether Q9's brand
# analysis is viable as written.
result = duckdb.sql("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN brand IS NULL THEN 1 ELSE 0 END) AS missing_brand,
        ROUND(100.0 * SUM(CASE WHEN brand IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_missing
    FROM '../data/events_with_sessions.parquet'
""")
result.show()

┌────────────┬───────────────┬─────────────┐
│ total_rows │ missing_brand │ pct_missing │
│   int64    │    int128     │   double    │
├────────────┼───────────────┼─────────────┤
│  109820004 │      15321303 │       13.95 │
└────────────┴───────────────┴─────────────┘



In [26]:
# Build dim_user: one row per user, with acquisition, activity and value attributes.
# We need data from two sources - all events (for activity dates) and purchases only
# (for order counts and revenue) - so we build each half separately then join.

duckdb.sql("""
    COPY (
        WITH activity AS (
            SELECT
                user_id,
                MIN(event_time) AS first_activity_at,
                MAX(event_time) AS last_activity_at,
                CAST(MIN(event_time) AS DATE) AS first_activity_date,
                CAST(MAX(event_time) AS DATE) AS last_activity_date
            FROM '../data/events_with_sessions.parquet'
            GROUP BY user_id
        ),
        purchases AS (
            SELECT
                user_id,
                MIN(event_time) AS first_purchase_at,
                MAX(event_time) AS last_purchase_at,
                CAST(MIN(event_time) AS DATE) AS first_purchase_date,
                CAST(MAX(event_time) AS DATE) AS last_purchase_date,
                COUNT(DISTINCT order_id) AS total_orders,
                COUNT(*) AS total_items_purchased,
                SUM(price) AS total_revenue
            FROM '../data/purchases_final.parquet'
            GROUP BY user_id
        )
        SELECT
            a.user_id,
            a.first_activity_at,
            a.first_activity_date,
            a.last_activity_at,
            a.last_activity_date,
            p.first_purchase_at,
            p.first_purchase_date,
            p.last_purchase_at,
            p.last_purchase_date,
            COALESCE(p.total_orders, 0) AS total_orders,
            COALESCE(p.total_items_purchased, 0) AS total_items_purchased,
            COALESCE(p.total_revenue, 0) AS total_revenue,
            CASE WHEN p.user_id IS NOT NULL THEN TRUE ELSE FALSE END AS has_purchased,
            DATE_TRUNC('week', p.first_purchase_date) AS acquisition_cohort_week
        FROM activity a
        LEFT JOIN purchases p ON a.user_id = p.user_id
    )
    TO '../data/dim_user.parquet'
    (FORMAT PARQUET)
""")

print("Done building dim_user")

Done building dim_user


In [27]:
duckdb.sql("""
    SELECT *
    FROM '../data/dim_user.parquet'
    LIMIT 10""")

┌───────────┬─────────────────────┬─────────────────────┬─────────────────────┬────────────────────┬─────────────────────┬─────────────────────┬─────────────────────┬────────────────────┬──────────────┬───────────────────────┬────────────────────┬───────────────┬─────────────────────────┐
│  user_id  │  first_activity_at  │ first_activity_date │  last_activity_at   │ last_activity_date │  first_purchase_at  │ first_purchase_date │  last_purchase_at   │ last_purchase_date │ total_orders │ total_items_purchased │   total_revenue    │ has_purchased │ acquisition_cohort_week │
│   int32   │      timestamp      │        date         │      timestamp      │        date        │      timestamp      │        date         │      timestamp      │        date        │    int64     │         int64         │       double       │    boolean    │        timestamp        │
├───────────┼─────────────────────┼─────────────────────┼─────────────────────┼────────────────────┼─────────────────────┼────────

In [28]:
# Verify dim_user: row count should match distinct users in the event data,
# and the purchaser/non-purchaser split should reconcile against what we
# already know from the repeat-purchase check.

result = duckdb.sql("""
    SELECT
        (SELECT COUNT(*) FROM '../data/dim_user.parquet') AS dim_user_rows,
        (SELECT COUNT(DISTINCT user_id) FROM '../data/events_with_sessions.parquet') AS distinct_users_in_events,
        (SELECT COUNT(*) FROM '../data/dim_user.parquet' WHERE has_purchased) AS purchasers,
        (SELECT COUNT(DISTINCT user_id) FROM '../data/purchases_final.parquet') AS distinct_users_in_purchases
""")

result.show()

┌───────────────┬──────────────────────────┬────────────┬─────────────────────────────┐
│ dim_user_rows │ distinct_users_in_events │ purchasers │ distinct_users_in_purchases │
│     int64     │          int64           │   int64    │            int64            │
├───────────────┼──────────────────────────┼────────────┼─────────────────────────────┤
│       5316649 │                  5316649 │     697470 │                      697470 │
└───────────────┴──────────────────────────┴────────────┴─────────────────────────────┘



In [29]:
# Verify total revenue reconciles against purchases_final
# (should match $451,807,232.56 from our earlier check)

result = duckdb.sql("""
    SELECT
        (SELECT SUM(total_revenue) FROM '../data/dim_user.parquet') AS revenue_from_dim_user,
        (SELECT SUM(price) FROM '../data/purchases_final.parquet') AS revenue_from_purchases_final
""")

result.show()

┌───────────────────────┬──────────────────────────────┐
│ revenue_from_dim_user │ revenue_from_purchases_final │
│        double         │            double            │
├───────────────────────┼──────────────────────────────┤
│     451807232.5601831 │            451807232.5601831 │
└───────────────────────┴──────────────────────────────┘



In [30]:
# Build fact_order_item: one row per (order, product) purchase, joined to
# the dim_product price version that was actually in effect at purchase time.
# This is a RANGE JOIN - matching not just on product_id, but on the purchase
# timestamp falling within the version's valid_from/valid_to window.

duckdb.sql("""
    COPY (
        SELECT
            p.order_id,
            p.user_id,
            p.product_id,
            p.category_id,
            p.event_time AS purchase_time,
            CAST(p.event_time AS DATE) AS purchase_date,
            p.rebuilt_session_id AS session_id,
            p.price AS event_price,
            d.price AS scd_price,
            d.brand,
            p.is_likely_non_standard_order
        FROM '../data/purchases_final.parquet' p
        LEFT JOIN '../data/dim_product.parquet' d
            ON p.product_id = d.product_id
            AND p.event_time >= d.valid_from
            AND (p.event_time < d.valid_to OR d.valid_to IS NULL)
    )
    TO '../data/fact_order_item.parquet'
    (FORMAT PARQUET)
""")

print("Done building fact_order_item")

Done building fact_order_item


In [31]:
# The critical check: did the range join accidentally multiply rows?
# If a purchase matched more than one price version, the count will be
# higher than 1,503,806 and we have a bug in the date-range logic.

result = duckdb.sql("""
    SELECT
        (SELECT COUNT(*) FROM '../data/purchases_final.parquet') AS source_rows,
        (SELECT COUNT(*) FROM '../data/fact_order_item.parquet') AS fact_rows
""")

result.show()

┌─────────────┬───────────┐
│ source_rows │ fact_rows │
│    int64    │   int64   │
├─────────────┼───────────┤
│     1503806 │   1503806 │
└─────────────┴───────────┘



In [32]:
result = duckdb.sql("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN scd_price IS NULL THEN 1 ELSE 0 END) AS no_price_version_matched,
        SUM(CASE WHEN event_price = scd_price THEN 1 ELSE 0 END) AS prices_match,
        SUM(CASE WHEN event_price != scd_price THEN 1 ELSE 0 END) AS prices_differ
    FROM '../data/fact_order_item.parquet'
""")

result.show()

┌────────────┬──────────────────────────┬──────────────┬───────────────┐
│ total_rows │ no_price_version_matched │ prices_match │ prices_differ │
│   int64    │          int128          │    int128    │    int128     │
├────────────┼──────────────────────────┼──────────────┼───────────────┤
│    1503806 │                        0 │      1502014 │          1792 │
└────────────┴──────────────────────────┴──────────────┴───────────────┘



In [33]:
# Look at the mismatches in detail. Comparing the purchase time against
# the version window it landed in should show what went wrong.

result = duckdb.sql("""
    SELECT
        product_id,
        purchase_time,
        event_price,
        scd_price,
        ROUND(ABS(event_price - scd_price), 2) AS price_gap
    FROM '../data/fact_order_item.parquet'
    WHERE event_price != scd_price
    ORDER BY purchase_time
    LIMIT 15
""")

result.show()

┌────────────┬─────────────────────┬─────────────┬───────────┬───────────┐
│ product_id │    purchase_time    │ event_price │ scd_price │ price_gap │
│   int32    │      timestamp      │    float    │   float   │   float   │
├────────────┼─────────────────────┼─────────────┼───────────┼───────────┤
│    1004767 │ 2019-10-10 04:30:48 │      250.93 │     250.2 │      0.73 │
│    4804056 │ 2019-10-10 04:39:07 │      161.87 │    161.88 │      0.01 │
│    1005008 │ 2019-10-10 04:47:47 │       98.59 │     98.84 │      0.25 │
│    1004750 │ 2019-10-10 05:21:28 │      197.12 │    197.13 │      0.01 │
│    1004665 │ 2019-10-10 05:38:27 │      857.42 │     857.6 │      0.18 │
│    1004838 │ 2019-10-10 07:49:55 │      179.28 │    179.22 │      0.06 │
│    1004873 │ 2019-10-10 07:59:37 │      379.11 │    379.08 │      0.03 │
│    1004792 │ 2019-10-10 08:26:23 │       486.5 │    476.18 │     10.32 │
│    4804056 │ 2019-10-10 09:36:27 │      161.87 │    157.76 │      4.11 │
│    1002524 │ 2019-10-10

In [34]:
# Test the boundary theory: for a mismatched purchase, look at all the
# price versions for that product around that time. If the purchase
# timestamp sits exactly on a version boundary, that's our answer.

result = duckdb.sql("""
    SELECT
        product_id,
        price,
        valid_from,
        valid_to
    FROM '../data/dim_product.parquet'
    WHERE product_id = 1004792
      AND valid_from BETWEEN '2019-10-10 08:00:00' AND '2019-10-10 09:00:00'
    ORDER BY valid_from
""")

result.show()

┌────────────┬────────┬─────────────────────┬─────────────────────┐
│ product_id │ price  │     valid_from      │      valid_to       │
│   int32    │ float  │      timestamp      │      timestamp      │
├────────────┼────────┼─────────────────────┼─────────────────────┤
│    1004792 │ 476.18 │ 2019-10-10 08:00:22 │ 2019-10-10 08:26:23 │
│    1004792 │ 476.18 │ 2019-10-10 08:26:23 │ 2019-10-10 08:26:28 │
│    1004792 │  486.5 │ 2019-10-10 08:26:23 │ 2019-10-10 08:26:23 │
│    1004792 │  486.5 │ 2019-10-10 08:26:28 │ 2019-10-10 11:23:49 │
└────────────┴────────┴─────────────────────┴─────────────────────┘



In [35]:
# How many dim_product rows have a broken window - either zero-length
# (valid_from = valid_to) or overlapping with the next version?

result = duckdb.sql("""
    SELECT
        COUNT(*) AS total_versions,
        SUM(CASE WHEN valid_from = valid_to THEN 1 ELSE 0 END) AS zero_length_windows
    FROM '../data/dim_product.parquet'
""")

result.show()

┌────────────────┬─────────────────────┐
│ total_versions │ zero_length_windows │
│     int64      │       int128        │
├────────────────┼─────────────────────┤
│         777246 │                2579 │
└────────────────┴─────────────────────┘



In [36]:
# Rebuild dim_product with two fixes:
# 1. Collapse to one price per (product, second) - taking the last price
#    recorded in that second - so same-second flapping can't create
#    zero-length or overlapping windows.
# 2. Add a deterministic tiebreaker to the ordering.

duckdb.sql("""
    COPY (
        WITH one_price_per_second AS (
            SELECT
                product_id,
                event_time,
                MAX(category_id) AS category_id,
                MAX(brand) AS brand,
                MAX(price) AS price
            FROM '../data/events_with_sessions.parquet'
            GROUP BY product_id, event_time
        ),
        price_events AS (
            SELECT
                *,
                LAG(price) OVER (PARTITION BY product_id ORDER BY event_time) AS previous_price
            FROM one_price_per_second
        ),
        change_points AS (
            SELECT product_id, category_id, brand, price, event_time AS valid_from
            FROM price_events
            WHERE previous_price IS NULL OR price != previous_price
        ),
        versioned AS (
            SELECT
                *,
                LEAD(valid_from) OVER (PARTITION BY product_id ORDER BY valid_from) AS valid_to,
                ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY valid_from DESC) AS recency_rank
            FROM change_points
        )
        SELECT
            product_id, category_id, brand, price, valid_from, valid_to,
            CASE WHEN recency_rank = 1 THEN TRUE ELSE FALSE END AS is_current
        FROM versioned
    )
    TO '../data/dim_product_v2.parquet'
    (FORMAT PARQUET)
""")

print("Done rebuilding dim_product with collision fix")

Done rebuilding dim_product with collision fix


In [37]:
# Compare old vs new: are the zero-length windows gone?

result = duckdb.sql("""
    SELECT
        (SELECT COUNT(*) FROM '../data/dim_product.parquet') AS old_versions,
        (SELECT COUNT(*) FROM '../data/dim_product_v2.parquet') AS new_versions,
        (SELECT SUM(CASE WHEN valid_from = valid_to THEN 1 ELSE 0 END) FROM '../data/dim_product.parquet') AS old_zero_length,
        (SELECT SUM(CASE WHEN valid_from = valid_to THEN 1 ELSE 0 END) FROM '../data/dim_product_v2.parquet') AS new_zero_length,
        (SELECT COUNT(DISTINCT product_id) FROM '../data/dim_product_v2.parquet') AS new_products
""")

result.show()

┌──────────────┬──────────────┬─────────────────┬─────────────────┬──────────────┐
│ old_versions │ new_versions │ old_zero_length │ new_zero_length │ new_products │
│    int64     │    int64     │     int128      │     int128      │    int64     │
├──────────────┼──────────────┼─────────────────┼─────────────────┼──────────────┤
│       777246 │       772840 │            2579 │               0 │       206876 │
└──────────────┴──────────────┴─────────────────┴─────────────────┴──────────────┘



In [38]:
# Rebuild fact_order_item against the corrected dim_product
duckdb.sql("""
    COPY (
        SELECT
            p.order_id, p.user_id, p.product_id, p.category_id,
            p.event_time AS purchase_time,
            CAST(p.event_time AS DATE) AS purchase_date,
            p.rebuilt_session_id AS session_id,
            p.price AS event_price,
            d.price AS scd_price,
            d.brand,
            p.is_likely_non_standard_order
        FROM '../data/purchases_final.parquet' p
        LEFT JOIN '../data/dim_product_v2.parquet' d
            ON p.product_id = d.product_id
            AND p.event_time >= d.valid_from
            AND (p.event_time < d.valid_to OR d.valid_to IS NULL)
    )
    TO '../data/fact_order_item_v2.parquet'
    (FORMAT PARQUET)
""")

result = duckdb.sql("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN scd_price IS NULL THEN 1 ELSE 0 END) AS unmatched,
        SUM(CASE WHEN event_price != scd_price THEN 1 ELSE 0 END) AS prices_differ
    FROM '../data/fact_order_item_v2.parquet'
""")

result.show()

┌────────────┬───────────┬───────────────┐
│ total_rows │ unmatched │ prices_differ │
│   int64    │  int128   │    int128     │
├────────────┼───────────┼───────────────┤
│    1503806 │         0 │          1802 │
└────────────┴───────────┴───────────────┘



In [39]:
result = duckdb.sql("""
    SELECT
        SUM(event_price) AS revenue_from_event_price,
        SUM(scd_price) AS revenue_from_scd_price
    FROM '../data/fact_order_item_v2.parquet'
""")
result.show()

┌──────────────────────────┬────────────────────────┐
│ revenue_from_event_price │ revenue_from_scd_price │
│          double          │         double         │
├──────────────────────────┼────────────────────────┤
│        451807232.5601831 │      451819906.9222608 │
└──────────────────────────┴────────────────────────┘



In [40]:
import os

# Retire the buggy versions, promote the fixed ones
os.remove("../data/dim_product.parquet")
os.remove("../data/fact_order_item.parquet")
os.rename("../data/dim_product_v2.parquet", "../data/dim_product.parquet")
os.rename("../data/fact_order_item_v2.parquet", "../data/fact_order_item.parquet")

print("Swapped in corrected versions")

Swapped in corrected versions


In [41]:
# Build fact_session_product: one row per product per visit, with flags
# for what happened to that product during that visit.
# This is the table the funnel questions need - session grain loses the
# product/category detail, and event grain is too fine to count cleanly.

duckdb.sql("""
    COPY (
        SELECT
            rebuilt_session_id AS session_id,
            user_id,
            product_id,
            category_id,
            CAST(MIN(event_time) AS DATE) AS session_date,
            MIN(event_time) AS first_event_time,
            MAX(event_time) AS last_event_time,
            SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS view_count,
            SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS cart_count,
            SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchase_count,
            SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) > 0 AS viewed,
            SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) > 0 AS carted,
            SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) > 0 AS purchased
        FROM '../data/events_with_sessions.parquet'
        GROUP BY rebuilt_session_id, user_id, product_id, category_id
    )
    TO '../data/fact_session_product.parquet'
    (FORMAT PARQUET)
""")

print("Done building fact_session_product")

Done building fact_session_product


In [42]:
# Check 1: row count should sit between the session count and the event count.
# Also confirm no rows were lost - total events across the table should
# reconcile back to the source.

result = duckdb.sql("""
    SELECT
        (SELECT COUNT(*) FROM '../data/fact_session_product.parquet') AS session_product_rows,
        (SELECT COUNT(DISTINCT rebuilt_session_id) FROM '../data/events_with_sessions.parquet') AS total_sessions,
        (SELECT COUNT(*) FROM '../data/events_with_sessions.parquet') AS total_events,
        (SELECT SUM(view_count + cart_count + purchase_count) FROM '../data/fact_session_product.parquet') AS events_accounted_for
""")

result.show()

┌──────────────────────┬────────────────┬──────────────┬──────────────────────┐
│ session_product_rows │ total_sessions │ total_events │ events_accounted_for │
│        int64         │     int64      │    int64     │        double        │
├──────────────────────┼────────────────┼──────────────┼──────────────────────┤
│             68023553 │       18776366 │    109820004 │          109820004.0 │
└──────────────────────┴────────────────┴──────────────┴──────────────────────┘



In [43]:
# Check 3: the funnel-order question Q1 asks about directly.
# How often does a purchase happen with no cart event in the same session?
# And how often with no view either?

result = duckdb.sql("""
    SELECT
        SUM(CASE WHEN purchased AND NOT carted THEN 1 ELSE 0 END) AS purchased_without_cart,
        SUM(CASE WHEN purchased AND NOT viewed THEN 1 ELSE 0 END) AS purchased_without_view,
        SUM(CASE WHEN carted AND NOT viewed THEN 1 ELSE 0 END) AS carted_without_view
    FROM '../data/fact_session_product.parquet'
""")

result.show()

┌────────────────────────┬────────────────────────┬─────────────────────┐
│ purchased_without_cart │ purchased_without_view │ carted_without_view │
│         int128         │         int128         │       int128        │
├────────────────────────┼────────────────────────┼─────────────────────┤
│                 507589 │                   9015 │               10654 │
└────────────────────────┴────────────────────────┴─────────────────────┘



In [44]:
result = duckdb.sql("""
    SELECT
        COUNT(*) AS total_product_sessions,
        SUM(CASE WHEN viewed THEN 1 ELSE 0 END) AS viewed,
        SUM(CASE WHEN carted THEN 1 ELSE 0 END) AS carted,
        SUM(CASE WHEN purchased THEN 1 ELSE 0 END) AS purchased
    FROM '../data/fact_session_product.parquet'
""")

result.show()

┌────────────────────────┬──────────┬─────────┬───────────┐
│ total_product_sessions │  viewed  │ carted  │ purchased │
│         int64          │  int128  │ int128  │  int128   │
├────────────────────────┼──────────┼─────────┼───────────┤
│               68023553 │ 68007734 │ 2534696 │   1503806 │
└────────────────────────┴──────────┴─────────┴───────────┘



In [2]:
# Build fact_session: one row per visit.
# Activity metrics come from the event table, but revenue comes from
# fact_order_item - because the event table still contains the repeat-purchase
# rows we excluded in Decision 6. Using the event table for revenue would
# give the inflated $503M figure instead of our agreed $451.8M.

duckdb.sql("""
    COPY (
        WITH session_activity AS (
            SELECT
                rebuilt_session_id AS session_id,
                user_id,
                MIN(event_time) AS session_start,
                MAX(event_time) AS session_end,
                CAST(MIN(event_time) AS DATE) AS session_date,
                DATE_DIFF('second', MIN(event_time), MAX(event_time)) / 60.0 AS duration_minutes,
                COUNT(*) AS total_events,
                SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS view_count,
                SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS cart_count,
                COUNT(DISTINCT product_id) AS distinct_products,
                COUNT(DISTINCT category_id) AS distinct_categories
            FROM '../data/events_with_sessions.parquet'
            GROUP BY rebuilt_session_id, user_id
        ),
        session_revenue AS (
            SELECT
                session_id,
                COUNT(*) AS items_purchased,
                SUM(event_price) AS revenue
            FROM '../data/fact_order_item.parquet'
            GROUP BY session_id
        )
        SELECT
            a.session_id,
            a.user_id,
            a.session_start,
            a.session_end,
            a.session_date,
            a.duration_minutes,
            a.total_events,
            a.view_count,
            a.cart_count,
            a.distinct_products,
            a.distinct_categories,
            COALESCE(r.items_purchased, 0) AS items_purchased,
            COALESCE(r.revenue, 0) AS revenue,
            a.cart_count > 0 AS had_cart,
            COALESCE(r.items_purchased, 0) > 0 AS had_purchase
        FROM session_activity a
        LEFT JOIN session_revenue r ON a.session_id = r.session_id
    )
    TO '../data/fact_session.parquet'
    (FORMAT PARQUET)
""")

print("Done building fact_session")

OutOfMemoryException: Out of Memory Error: could not allocate block of size 256.0 KiB (12.5 GiB/12.5 GiB used)

Possible solutions:
* Reducing the number of threads (SET threads=X)
* Disabling insertion-order preservation (SET preserve_insertion_order=false)
* Increasing the memory limit (SET memory_limit='...GB')

See also https://duckdb.org/docs/stable/guides/performance/how_to_tune_workloads

In [46]:
# Two settings to help DuckDB cope with this heavier query:
# - preserve_insertion_order=false lets DuckDB reorder rows freely, which
#   frees up a lot of memory. We don't care about row order in the output.
# - a temp directory lets it spill to disk instead of failing when memory runs out.

duckdb.sql("SET preserve_insertion_order=false")
duckdb.sql("SET temp_directory='D:/duckdb_temp'")

print("Settings applied")

NotImplementedException: Not implemented Error: Cannot switch temporary directory after the current one has been used

In [1]:
import duckdb

duckdb.sql("SET temp_directory='D:/duckdb_temp'")
duckdb.sql("SET preserve_insertion_order=false")
duckdb.sql("SET threads=4")

print("Settings applied")

Settings applied


In [3]:
# Build the activity half first, WITHOUT the expensive COUNT(DISTINCT) columns.
# Those are what's blowing up memory - DuckDB has to track every unique value
# per session across 18.8M sessions simultaneously.

duckdb.sql("""
    COPY (
        SELECT
            rebuilt_session_id AS session_id,
            user_id,
            MIN(event_time) AS session_start,
            MAX(event_time) AS session_end,
            CAST(MIN(event_time) AS DATE) AS session_date,
            DATE_DIFF('second', MIN(event_time), MAX(event_time)) / 60.0 AS duration_minutes,
            COUNT(*) AS total_events,
            SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS view_count,
            SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS cart_count
        FROM '../data/events_with_sessions.parquet'
        GROUP BY rebuilt_session_id, user_id
    )
    TO '../data/_session_activity.parquet'
    (FORMAT PARQUET)
""")

print("Step 1 done: activity half")

Step 1 done: activity half


In [4]:
# Distinct product/category counts from fact_session_product.
# Because that table already has one row per session-product, counting
# products is just COUNT(*) - no expensive DISTINCT tracking needed.

duckdb.sql("""
    COPY (
        SELECT
            session_id,
            COUNT(*) AS distinct_products,
            COUNT(DISTINCT category_id) AS distinct_categories
        FROM '../data/fact_session_product.parquet'
        GROUP BY session_id
    )
    TO '../data/_session_breadth.parquet'
    (FORMAT PARQUET)
""")

print("Step 2 done: breadth counts")

Step 2 done: breadth counts


In [5]:
# Step 3: join the three summarised pieces together.
# All inputs are already at session level, so this is a light join
# rather than a heavy aggregation.

duckdb.sql("""
    COPY (
        WITH session_revenue AS (
            SELECT
                session_id,
                COUNT(*) AS items_purchased,
                SUM(event_price) AS revenue
            FROM '../data/fact_order_item.parquet'
            GROUP BY session_id
        )
        SELECT
            a.session_id,
            a.user_id,
            a.session_start,
            a.session_end,
            a.session_date,
            a.duration_minutes,
            a.total_events,
            a.view_count,
            a.cart_count,
            COALESCE(b.distinct_products, 0) AS distinct_products,
            COALESCE(b.distinct_categories, 0) AS distinct_categories,
            COALESCE(r.items_purchased, 0) AS items_purchased,
            COALESCE(r.revenue, 0) AS revenue,
            a.cart_count > 0 AS had_cart,
            COALESCE(r.items_purchased, 0) > 0 AS had_purchase
        FROM '../data/_session_activity.parquet' a
        LEFT JOIN '../data/_session_breadth.parquet' b ON a.session_id = b.session_id
        LEFT JOIN session_revenue r ON a.session_id = r.session_id
    )
    TO '../data/fact_session.parquet'
    (FORMAT PARQUET)
""")

print("Done building fact_session")

Done building fact_session


In [6]:
result = duckdb.sql("""
    SELECT
        COUNT(*) AS session_rows,
        SUM(revenue) AS total_revenue,
        SUM(CASE WHEN had_purchase THEN 1 ELSE 0 END) AS sessions_with_purchase
    FROM '../data/fact_session.parquet'
""")

result.show()

┌──────────────┬───────────────────┬────────────────────────┐
│ session_rows │   total_revenue   │ sessions_with_purchase │
│    int64     │      double       │         int128         │
├──────────────┼───────────────────┼────────────────────────┤
│     18776366 │ 451807232.5601831 │                1291108 │
└──────────────┴───────────────────┴────────────────────────┘



In [7]:
import os

for f in ["_session_activity.parquet", "_session_breadth.parquet"]:
    path = f"../data/{f}"
    if os.path.exists(path):
        os.remove(path)
        print(f"Removed {f}")

Removed _session_activity.parquet
Removed _session_breadth.parquet


In [3]:
import os
os.makedirs("../data/bi", exist_ok=True)
print(os.path.exists("../data/bi"))

True


In [4]:
import duckdb
duckdb.sql("SET preserve_insertion_order=false")

duckdb.sql("""
    COPY (
        SELECT
            session_date AS date_key,
            COUNT(*) AS sessions,
            SUM(CASE WHEN had_purchase THEN 1 ELSE 0 END) AS purchasing_sessions,
            SUM(CASE WHEN had_cart THEN 1 ELSE 0 END) AS sessions_with_cart,
            SUM(revenue) AS revenue
        FROM '../data/fact_session.parquet'
        GROUP BY session_date
    )
    TO '../data/bi/agg_daily_sessions.csv'
    (FORMAT CSV, HEADER)
""")

In [5]:
import duckdb

result = duckdb.sql("""
    SELECT
        COUNT(*) AS days,
        SUM(sessions) AS total_sessions,
        SUM(purchasing_sessions) AS purchasing_sessions,
        ROUND(SUM(revenue), 2) AS total_revenue
    FROM '../data/bi/agg_daily_sessions.csv'
""")

result.show()

┌───────┬────────────────┬─────────────────────┬───────────────┐
│ days  │ total_sessions │ purchasing_sessions │ total_revenue │
│ int64 │     int128     │       int128        │    double     │
├───────┼────────────────┼─────────────────────┼───────────────┤
│    61 │       18776366 │             1291108 │  451807232.56 │
└───────┴────────────────┴─────────────────────┴───────────────┘



In [6]:
# Revenue and items by day and category.
# Deliberately excludes order and buyer counts: an order spanning two
# categories would count in both, so summing across categories would
# overstate the total. Those figures come from agg_daily_sessions.

duckdb.sql("""
    COPY (
        SELECT
            purchase_date AS date_key,
            category_id,
            COUNT(*) AS items,
            SUM(event_price) AS revenue
        FROM '../data/fact_order_item.parquet'
        GROUP BY purchase_date, category_id
    )
    TO '../data/bi/agg_daily_category.csv'
    (FORMAT CSV, HEADER)
""")

print("Done")

Done


In [7]:
result = duckdb.sql("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT date_key) AS days,
        COUNT(DISTINCT category_id) AS categories,
        SUM(items) AS total_items,
        ROUND(SUM(revenue), 2) AS total_revenue
    FROM '../data/bi/agg_daily_category.csv'
""")

result.show()

┌───────┬───────┬────────────┬─────────────┬───────────────┐
│ rows  │ days  │ categories │ total_items │ total_revenue │
│ int64 │ int64 │   int64    │   int128    │    double     │
├───────┼───────┼────────────┼─────────────┼───────────────┤
│ 24360 │    60 │        636 │     1503806 │  451807232.56 │
└───────┴───────┴────────────┴─────────────┴───────────────┘



In [8]:
# Funnel counts by day, category and brand.
# Storing COUNTS not rates - rates cannot be summed or averaged across
# hierarchy levels, but counts can, so Power BI recomputes the ratio
# correctly at whatever level is being viewed.
# Note carted_and_purchased, NOT purchased: the cart-to-purchase rate must
# only count journeys that actually passed through the cart.

duckdb.sql("""
    COPY (
        WITH product_brand AS (
            SELECT product_id, brand
            FROM (
                SELECT product_id, brand,
                    ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY COUNT(*) DESC) AS rn
                FROM '../data/events_with_sessions.parquet'
                WHERE brand IS NOT NULL
                GROUP BY product_id, brand
            ) WHERE rn = 1
        )
        SELECT
            f.session_date AS date_key,
            f.category_id,
            COALESCE(b.brand, 'unknown') AS brand,
            SUM(CASE WHEN f.viewed THEN 1 ELSE 0 END) AS viewed,
            SUM(CASE WHEN f.carted THEN 1 ELSE 0 END) AS carted,
            SUM(CASE WHEN f.carted AND f.purchased THEN 1 ELSE 0 END) AS carted_and_purchased,
            SUM(CASE WHEN f.purchased THEN 1 ELSE 0 END) AS purchased_any
        FROM '../data/fact_session_product.parquet' f
        LEFT JOIN product_brand b ON f.product_id = b.product_id
        GROUP BY f.session_date, f.category_id, COALESCE(b.brand, 'unknown')
    )
    TO '../data/bi/agg_funnel.csv'
    (FORMAT CSV, HEADER)
""")

print("Done")

Done


In [9]:
result = duckdb.sql("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT date_key) AS days,
        COUNT(DISTINCT category_id) AS categories,
        COUNT(DISTINCT brand) AS brands,
        SUM(viewed) AS total_viewed,
        SUM(carted) AS total_carted,
        SUM(carted_and_purchased) AS total_carted_and_purchased,
        SUM(purchased_any) AS total_purchased
    FROM '../data/bi/agg_funnel.csv'
""")

result.show()

┌────────┬───────┬────────────┬────────┬──────────────┬──────────────┬────────────────────────────┬─────────────────┐
│  rows  │ days  │ categories │ brands │ total_viewed │ total_carted │ total_carted_and_purchased │ total_purchased │
│ int64  │ int64 │   int64    │ int64  │    int128    │    int128    │           int128           │     int128      │
├────────┼───────┼────────────┼────────┼──────────────┼──────────────┼────────────────────────────┼─────────────────┤
│ 445447 │    61 │        691 │   4297 │     68007734 │      2534696 │                     996217 │         1503806 │
└────────┴───────┴────────────┴────────┴──────────────┴──────────────┴────────────────────────────┴─────────────────┘



In [10]:
# Product totals across the whole period, for Q9's ranking, ABC
# classification and price bands.

duckdb.sql("""
    COPY (
        SELECT
            product_id,
            category_id,
            COUNT(*) AS items_sold,
            COUNT(DISTINCT order_id) AS orders,
            SUM(event_price) AS revenue,
            ROUND(AVG(event_price), 2) AS avg_price
        FROM '../data/fact_order_item.parquet'
        GROUP BY product_id, category_id
    )
    TO '../data/bi/agg_product_totals.csv'
    (FORMAT CSV, HEADER)
""")

print("Done")

Done


In [11]:
# One row per purchasing customer per week in which they placed an order.
# This is the raw material for the cohort grid - deliberately NOT
# pre-aggregated, so the cohort logic can be built in DAX.

duckdb.sql("""
    COPY (
        SELECT DISTINCT
            user_id,
            DATE_TRUNC('week', purchase_date) AS order_week
        FROM '../data/fact_order_item.parquet'
    )
    TO '../data/bi/agg_user_weekly_orders.csv'
    (FORMAT CSV, HEADER)
""")

print("Done")

Done


In [12]:
result = duckdb.sql("""
    SELECT
        COUNT(*) AS products,
        SUM(items_sold) AS total_items,
        ROUND(SUM(revenue), 2) AS total_revenue
    FROM '../data/bi/agg_product_totals.csv'
""")
result.show()

┌──────────┬─────────────┬───────────────┐
│ products │ total_items │ total_revenue │
│  int64   │   int128    │    double     │
├──────────┼─────────────┼───────────────┤
│    68079 │     1503806 │  451807232.56 │
└──────────┴─────────────┴───────────────┘



In [13]:
result = duckdb.sql("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT user_id) AS customers,
        COUNT(DISTINCT order_week) AS weeks
    FROM '../data/bi/agg_user_weekly_orders.csv'
""")
result.show()

┌────────┬───────────┬───────┐
│  rows  │ customers │ weeks │
│ int64  │   int64   │ int64 │
├────────┼───────────┼───────┤
│ 979709 │    697470 │     9 │
└────────┴───────────┴───────┘



In [14]:
# dim_time: one row per hour, with daypart labels.
# Hours are shown in inferred local time (UTC+3) - see Q2.

duckdb.sql("""
    COPY (
        SELECT
            h AS hour_utc,
            (h + 3) % 24 AS hour_local,
            CASE
                WHEN (h + 3) % 24 BETWEEN 0 AND 5 THEN 'Night'
                WHEN (h + 3) % 24 BETWEEN 6 AND 11 THEN 'Morning'
                WHEN (h + 3) % 24 BETWEEN 12 AND 17 THEN 'Afternoon'
                ELSE 'Evening'
            END AS daypart
        FROM generate_series(0, 23) AS t(h)
    )
    TO '../data/bi/dim_time.csv'
    (FORMAT CSV, HEADER)
""")

# dim_period_offset: the numbers 0-8, for the retention grid columns.
# Disconnected table - no relationship to anything else in the model.

duckdb.sql("""
    COPY (
        SELECT n AS period_offset
        FROM generate_series(0, 8) AS t(n)
    )
    TO '../data/bi/dim_period_offset.csv'
    (FORMAT CSV, HEADER)
""")

print("Done")

Done


In [15]:
# dim_user: purchasers only (697,470 not 5.3m)
duckdb.sql("""
    COPY (SELECT * FROM '../data/dim_user.parquet' WHERE has_purchased)
    TO '../data/bi/dim_user.csv' (FORMAT CSV, HEADER)
""")

# dim_product: current price only (206,876 not 772,840)
duckdb.sql("""
    COPY (SELECT * FROM '../data/dim_product.parquet' WHERE is_current)
    TO '../data/bi/dim_product.csv' (FORMAT CSV, HEADER)
""")

duckdb.sql("""
    COPY (SELECT * FROM '../data/dim_category.parquet')
    TO '../data/bi/dim_category.csv' (FORMAT CSV, HEADER)
""")

duckdb.sql("""
    COPY (SELECT * FROM '../data/dim_date.parquet')
    TO '../data/bi/dim_date.csv' (FORMAT CSV, HEADER)
""")

print("Done")

Done


In [16]:
import os

bi_folder = "../data/bi"
files = sorted(os.listdir(bi_folder))

total_mb = 0
for f in files:
    size_mb = os.path.getsize(os.path.join(bi_folder, f)) / (1024**2)
    total_mb += size_mb
    print(f"{f:<35} {size_mb:>8.2f} MB")

print(f"\n{'TOTAL':<35} {total_mb:>8.2f} MB")

agg_daily_category.csv                  1.20 MB
agg_daily_sessions.csv                  0.00 MB
agg_funnel.csv                         20.12 MB
agg_product_totals.csv                  3.73 MB
agg_user_weekly_orders.csv             28.03 MB
dim_category.csv                        0.05 MB
dim_date.csv                            0.00 MB
dim_period_offset.csv                   0.00 MB
dim_product.csv                        13.22 MB
dim_time.csv                            0.00 MB
dim_user.csv                          120.44 MB

TOTAL                                 186.80 MB


In [17]:
# Slim dim_user: keep only what the report actually uses.
# The four full-precision timestamp columns have millions of distinct
# values and compress badly - a classic expensive-column problem.

duckdb.sql("""
    COPY (
        SELECT
            user_id,
            first_purchase_date,
            last_purchase_date,
            acquisition_cohort_week,
            total_orders,
            total_items_purchased,
            total_revenue
        FROM '../data/dim_user.parquet'
        WHERE has_purchased
    )
    TO '../data/bi/dim_user.csv' (FORMAT CSV, HEADER)
""")

print("Done - re-check the size")

Done - re-check the size


In [18]:
import os

bi_folder = "../data/bi"
total_mb = 0
for f in sorted(os.listdir(bi_folder)):
    size_mb = os.path.getsize(os.path.join(bi_folder, f)) / (1024**2)
    total_mb += size_mb
    print(f"{f:<35} {size_mb:>8.2f} MB")

print(f"\n{'TOTAL':<35} {total_mb:>8.2f} MB")

agg_daily_category.csv                  1.20 MB
agg_daily_sessions.csv                  0.00 MB
agg_funnel.csv                         20.12 MB
agg_product_totals.csv                  3.73 MB
agg_user_weekly_orders.csv             28.03 MB
dim_category.csv                        0.05 MB
dim_date.csv                            0.00 MB
dim_period_offset.csv                   0.00 MB
dim_product.csv                        13.22 MB
dim_time.csv                            0.00 MB
dim_user.csv                           49.27 MB

TOTAL                                 115.62 MB


In [19]:
# agg_product_totals without category_id - the category comes via
# dim_product, so carrying it here creates two paths between the tables
# and forces Power BI to deactivate one.

duckdb.sql("""
    COPY (
        SELECT
            product_id,
            COUNT(*) AS items_sold,
            COUNT(DISTINCT order_id) AS orders,
            SUM(event_price) AS revenue,
            ROUND(AVG(event_price), 2) AS avg_price
        FROM '../data/fact_order_item.parquet'
        GROUP BY product_id
    )
    TO '../data/bi/agg_product_totals.csv'
    (FORMAT CSV, HEADER)
""")

In [20]:
import duckdb

# ABC classification computed with a window function rather than DAX.
# A running total is one pass in SQL; in DAX it must be recalculated
# for every visible row, which exceeded available resources at this scale.
# Reading from the parquet source rather than the CSV, since we can't
# read and write the same CSV in one statement.

duckdb.sql("""
    COPY (
        WITH totals AS (
            SELECT
                product_id,
                COUNT(*) AS items_sold,
                COUNT(DISTINCT order_id) AS orders,
                SUM(event_price) AS revenue,
                ROUND(AVG(event_price), 2) AS avg_price
            FROM '../data/fact_order_item.parquet'
            GROUP BY product_id
        )
        SELECT
            *,
            SUM(revenue) OVER (ORDER BY revenue DESC) AS cumulative_revenue,
            SUM(revenue) OVER (ORDER BY revenue DESC) / SUM(revenue) OVER () AS cumulative_pct,
            ROW_NUMBER() OVER (ORDER BY revenue DESC) AS revenue_rank,
            CASE
                WHEN SUM(revenue) OVER (ORDER BY revenue DESC) / SUM(revenue) OVER () <= 0.80 THEN 'A'
                WHEN SUM(revenue) OVER (ORDER BY revenue DESC) / SUM(revenue) OVER () <= 0.95 THEN 'B'
                ELSE 'C'
            END AS abc_class
        FROM totals
    )
    TO '../data/bi/agg_product_totals.csv'
    (FORMAT CSV, HEADER)
""")

print("Done")

Done


In [21]:
result = duckdb.sql("""
    SELECT
        abc_class,
        COUNT(*) AS products,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_products,
        ROUND(SUM(revenue), 2) AS revenue,
        ROUND(100.0 * SUM(revenue) / SUM(SUM(revenue)) OVER (), 2) AS pct_of_revenue
    FROM '../data/bi/agg_product_totals.csv'
    GROUP BY abc_class
    ORDER BY abc_class
""")
result.show()

┌───────────┬──────────┬─────────────────┬──────────────┬────────────────┐
│ abc_class │ products │ pct_of_products │   revenue    │ pct_of_revenue │
│  varchar  │  int64   │     double      │    double    │     double     │
├───────────┼──────────┼─────────────────┼──────────────┼────────────────┤
│ A         │      727 │            1.07 │ 361400540.87 │          79.99 │
│ B         │     7546 │           11.08 │  67814047.09 │          15.01 │
│ C         │    59803 │           87.85 │   22592644.6 │            5.0 │
└───────────┴──────────┴─────────────────┴──────────────┴────────────────┘



In [22]:
import duckdb
result = duckdb.sql("SELECT * FROM '../data/bi/agg_product_totals.csv' LIMIT 3")
result.show()

┌────────────┬────────────┬────────┬────────────────────┬───────────┬────────────────────┬─────────────────────┬──────────────┬───────────┐
│ product_id │ items_sold │ orders │      revenue       │ avg_price │ cumulative_revenue │   cumulative_pct    │ revenue_rank │ abc_class │
│   int64    │   int64    │ int64  │       double       │  double   │       double       │       double        │    int64     │  varchar  │
├────────────┼────────────┼────────┼────────────────────┼───────────┼────────────────────┼─────────────────────┼──────────────┼───────────┤
│    1005115 │      31035 │  31035 │  29465381.73284912 │    949.42 │  29465381.73284912 │ 0.06521671104263294 │            1 │ A         │
│    1005105 │      14227 │  14227 │ 19551518.377685547 │   1374.25 │  49016900.11053467 │   0.108490738036172 │            2 │ A         │
│    1004249 │      15720 │  15720 │  11845716.66192627 │    753.54 │  60862616.77246094 │  0.1347092573697428 │            3 │ A         │
└────────────┴──────